# Step 1: Data Preparation
Load, clean, and normalize the loan dataset for training.

In [ ]:
import pandas as pd
import boto3
import os
from io import StringIO

S3_ENDPOINT = os.environ.get("S3_ENDPOINT", "http://minio:9000")
S3_BUCKET = os.environ.get("S3_BUCKET", "pipelines")
AWS_ACCESS_KEY_ID = os.environ.get("AWS_ACCESS_KEY_ID", "minio")
AWS_SECRET_ACCESS_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY", "minio123")

s3 = boto3.client("s3", endpoint_url=S3_ENDPOINT,
                   aws_access_key_id=AWS_ACCESS_KEY_ID,
                   aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
                   verify=False)

DATA_KEY = os.environ.get("DATA_KEY", "data/sample-loans.csv")
OUTPUT_KEY = os.environ.get("OUTPUT_KEY", "pipeline-artifacts/cleaned-data.csv")

obj = s3.get_object(Bucket=S3_BUCKET, Key=DATA_KEY)
df = pd.read_csv(obj["Body"])
print(f"Loaded {len(df)} rows, {len(df.columns)} columns from s3://{S3_BUCKET}/{DATA_KEY}")
df.head()

In [ ]:
df = df.dropna().drop_duplicates()

numeric_cols = ["age", "income", "loan_amount", "interest_rate",
                "credit_score", "dti_ratio", "employment_length",
                "num_credit_lines", "delinquencies"]

for col in numeric_cols:
    if col in df.columns:
        mean_val = df[col].mean()
        std_val = df[col].std()
        if std_val > 0:
            df[col] = (df[col] - mean_val) / std_val

cat_cols = ["home_ownership", "loan_purpose"]
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print(f"Cleaned: {len(df)} rows, {len(df.columns)} columns")

In [ ]:
csv_buffer = StringIO()
df.to_csv(csv_buffer, index=False)
s3.put_object(Bucket=S3_BUCKET, Key=OUTPUT_KEY, Body=csv_buffer.getvalue())
print(f"Saved to s3://{S3_BUCKET}/{OUTPUT_KEY}")